# 525 Talent Pool Analysis

**Purpose**: Use 520 (and other) outputs to find mechanisms besides OER content that align with strong talent pools — **who** (senior raters) and **where** (units/divisions).

**Inputs**: 520 cell outputs (e.g. Cell 7 for div_name), UIC tables. Pool strength = **pool means** (mean TB in pool).

**Layout**: See `current_documents/525_plans.md` §5.
---

## 1. Setup and design note

- Senior raters first; rater-ready (parameter to add later).
- UIC from existing tables; 525 only reads/aggregates.
- Start with **division-level** aggregation; brigade/battalion later.
- Recompute when needed → use a **separate .py** callable from here.

In [ ]:
# === 1. SETUP ===
# Imports, paths, and any 525-specific config (e.g. include_lt_rated_oers)
from functionsG import load_feather, store_feather  # adjust to your loader
import pandas as pd
import numpy as np

# 525 options (stub)
INCLUDE_LT_RATED_OERS = True   # Include snapshot rows where rated officer is LT (senior rater view)
POOL_STRENGTH_METRIC = 'mean_tb'  # Use pool means for 'strongest' pools (not pool-minus-mean)

print('525 setup: senior raters first; division-level first.')

## 2. Load inputs

Load **specific 520 cell outputs** as needed. For division work: **520 Cell 7** (e.g. `df_pipeline_07_time_varying`) has `div_name`. UIC tables for brigade/battalion when we add them. LT OER data is already on the snapshot frame in 520 (04a–09).

In [ ]:
# === 2. LOAD INPUTS ===
# 520 Cell 7 output: snapshot-level with div_name, pool metrics, senior rater id
df_525_base = load_feather('df_pipeline_07_time_varying')  # or 09_filtered if post-filter desired

# Optional: UIC tables for brigade/battalion (when we add that step)
# df_uic_hierarchy = load_feather('df_uic_hierarchy')
# df_uic_div_lookup = load_feather('df_uic_div_lookup')

print(f'Loaded base: {len(df_525_base):,} rows, {df_525_base.pid_pde.nunique():,} officers')
print('Columns (sample):', [c for c in df_525_base.columns if 'pool' in c.lower() or c == 'div_name'][:15])

## 3. Define "strongest senior-rating pools"

Single definition everywhere. Use **pool means** (mean TB in pool) to rank pools — not pool-minus-mean. E.g. top decile of pools by mean TB, or pools above a threshold.

In [ ]:
# === 3. DEFINE STRONGEST POOLS ===
# Build pool-level mean TB (if not already in 520 output); then define threshold or top fraction.
# Stub: assume 520 already has a pool id and mean TB per pool; we define 'strong' here.

POOL_MEAN_TB_COL = 'pool_mean_tb_snr_fwd'   # adjust to actual 520 column name
STRONG_POOL_TOP_DECILE = True              # True = top 10% of pools by mean TB; else use threshold
STRONG_POOL_THRESHOLD = None               # e.g. 0.65 if not using decile

# TODO: compute pool-level mean TB if needed; then label pools as strong/not
# df_pool_means = df_525_base.groupby('pool_id_snr_fwd')[POOL_MEAN_TB_COL].agg('first').reset_index()
# strong_threshold = df_pool_means[POOL_MEAN_TB_COL].quantile(0.9) if STRONG_POOL_TOP_DECILE else STRONG_POOL_THRESHOLD

print('Stub: define strong pools by pool mean TB (column name TBD from 520).')

## 4. UIC analysis (division-level first)

Aggregate **pool strength** (or share of high-TB ratings) by **division** using `div_name` from the loaded frame. Report/plot which divisions are associated with strongest senior-rating pools. Brigade/battalion later.

In [ ]:
# === 4. UIC ANALYSIS — DIVISION LEVEL ===
# Each row is a snapshot; we have div_name (rated officer's division), pool id, pool mean TB.
# Aggregate: by div_name, compute mean pool strength or share of snapshots in strong pools.

if 'div_name' not in df_525_base.columns:
    print('Warning: div_name missing. Run 520 with DIVISION_CONFIG["enabled"]=True (e.g. Cell 7).')
else:
    # Stub: division × pool strength (e.g. mean of pool mean TB per division)
    # div_strength = df_525_base.groupby('div_name').agg(...)
    print('Stub: aggregate pool strength by div_name; plot/report top divisions.')
    print('Division value counts (sample):', df_525_base['div_name'].value_counts().head())

## 5. Senior rater pid_pde analysis

Which senior rater `pid_pde`s appear most often in high-TB (strongest) pools? **Switch**: `INCLUDE_LT_RATED_OERS` — include or exclude snapshot rows where the **rated** officer is LT (same COLs as LTC/BN commanders rating LTs).

In [ ]:
# === 5. SENIOR RATER ANALYSIS ===
# Optionally restrict to CPT (and MAJ) snapshot rows only, or include LT rows.
RANK_COL = 'rank_pde'  # or whatever 520 uses
LT_RANKS = ['2LT', '1LT', 'O01', 'O02']  # adjust to match 520

if not INCLUDE_LT_RATED_OERS:
    df_snr = df_525_base[~df_525_base[RANK_COL].isin(LT_RANKS)].copy()
else:
    df_snr = df_525_base.copy()

# Stub: which senior rater pid_pde(s) appear most in strong pools?
# sr_in_strong = df_snr[df_snr['in_strong_pool']].groupby('snr_rater_pid_pde').size().sort_values(ascending=False)
print(f'Senior rater analysis: {"including" if INCLUDE_LT_RATED_OERS else "excluding"} LT-rated rows')
print(f'Rows: {len(df_snr):,}')

## 6. Prestige / interpretation

Short section: UICs and senior rater persistence as candidate mechanisms that correlate with OER-based talent pools and tie to prestige.

In [ ]:
# === 6. PRESTIGE / INTERPRETATION ===
# Summarize: which divisions / senior raters line up with strongest pools; narrative stub.
print('Stub: narrative and summary stats linking divisions and senior raters to strong pools.')

## 7. Extensibility

Where to set `rater_role` (or equivalent) for adding **rater** (not just senior rater) later. Where to add senior rater **UIC** analysis if desired.

In [ ]:
# === 7. EXTENSIBILITY ===
RATER_ROLE = 'senior_rater'  # 'rater' when we add rater-level pools
# Senior rater UIC: same aggregation logic as rated officer UIC, keyed by senior rater's unit.
print('Stub: rater_role =', RATER_ROLE, '; senior rater UIC placeholder.')

## 8. UICs and longevity

Which UICs (division, then brigade/battalion) are associated with greatest longevity? Longevity = time from commissioning to exit. Use same UIC tables and categorization.

In [ ]:
# === 8. UICS AND LONGEVITY ===
# Requires: commissioning/appointment date, exit date; optional UIC at each snapshot.
# Stub: merge longevity and UIC; summarize mean/median tenure by division (then brigade/battalion).
print('Stub: longevity by division (and later brigade/battalion).')

## 9. UIC categorization (division, function, etc.)

Join UICs to division, function, or other type. Summaries and plots by category (e.g. talent-pool strength by division, longevity by function).

In [ ]:
# === 9. UIC CATEGORIZATION ===
# Division already in use; add function (combat arms / support / etc.) if we have a lookup.
print('Stub: categorize UICs by type; summaries by category.')

## 10. Path-through-network analysis

For officers who achieve **highest rank** and **longevity**: build their **sequence** of unit types (by division, function, or UIC) over time. Describe patterns (e.g. common sequences, modal paths).

In [ ]:
# === 10. PATH-THROUGH-NETWORK ===
# Per officer: order snapshots by time; sequence of div_name (or function).
# Restrict to high-rank, long-tenure officers; summarize modal paths.
print('Stub: path-through-network for high-rank, long-tenure officers.')